# Air Quality Analytics Report

This notebook summarizes the curated Sofia AQI dataset using the shared Spark batch analytics pipeline in `src/analytics/`.

It loads the report tables once, converts them to pandas for presentation, and closes the Spark session automatically.

## Source and Report Scope

Curated source path:

`hdfs://namenode:9000/data/air-quality/<CITY>/curated/*.jsonl`

Report tables included in this notebook:

- `hourly_aqi`
- `daily_aqi`
- `aqi_category_distribution`
- `average_pollutants`
- `dominant_pollutants`
- `weather_correlations`

In [ ]:
from IPython.display import display
import os
import numpy as np
from matplotlib import pyplot as plt

from src.analytics.batch_analysis import DEFAULT_HDFS_ROOT
from src.analytics.batch_analysis import create_spark_session
from src.analytics.batch_analysis import run_batch_analysis

plt.style.use("seaborn-v0_8-whitegrid")


def load_report_tables():
    spark = create_spark_session("air-quality-analytics-notebook")
    try:
        city = os.getenv("CITY")
        hdfs_root = os.getenv("OUTPUT_ROOT") or DEFAULT_HDFS_ROOT
        if not city:
            raise ValueError("CITY is required")
        results = run_batch_analysis(spark, hdfs_root, city)
        normalized = results["normalized"]
        return {
            "normalized_schema": normalized._jdf.schema().treeString(),
            "normalized_count": normalized.count(),
            "normalized_preview": (
                normalized.select(
                    "timestamp",
                    "event_timestamp",
                    "day",
                    "hour",
                    "aqi",
                    "dominant_pollutant",
                )
                .limit(5)
                .toPandas()
            ),
            "hourly_aqi": results["hourly_aqi"].toPandas(),
            "daily_aqi": results["daily_aqi"].toPandas(),
            "aqi_category_distribution": results[
                "aqi_category_distribution"
            ].toPandas(),
            "average_pollutants": results["average_pollutants"].toPandas(),
            "dominant_pollutants": results["dominant_pollutants"].toPandas(),
            "weather_correlations": results["weather_correlations"].toPandas(),
        }
    finally:
        spark.stop()


report_tables = load_report_tables()
sorted(report_tables)

## Dataset Snapshot

In [ ]:
print(report_tables["normalized_schema"])
print(f"Normalized records included in analytics: {report_tables['normalized_count']}")
display(report_tables["normalized_preview"])

## AQI Trends

In [ ]:
hourly_pdf = report_tables["hourly_aqi"]
display(hourly_pdf)

In [ ]:
ax = hourly_pdf.plot(
    kind="line", x="hour", y="avg_aqi", marker="o", legend=False, figsize=(8, 4)
)
ax.set_title("Average AQI by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Average AQI")
plt.tight_layout()
plt.show()

In [ ]:
daily_pdf = report_tables["daily_aqi"]
display(daily_pdf)

In [ ]:
ax = daily_pdf.plot(
    kind="line",
    x="day",
    y="avg_aqi",
    marker="o",
    legend=False,
    figsize=(9, 4),
    color="orange",
)
ax.set_title("Average AQI by Day")
ax.set_xlabel("Day")
ax.set_ylabel("Average AQI")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## AQI Category Distribution

In [ ]:
aqi_category_pdf = report_tables["aqi_category_distribution"]
display(aqi_category_pdf)

In [ ]:
colors_map = {
    "Good": "green",
    "Moderate": "gold",
    "Unhealthy for Sensitive Groups": "orange",
    "Unhealthy": "red",
    "Very Unhealthy": "purple",
    "Hazardous": "maroon",
}
colors = [colors_map.get(cat, "gray") for cat in aqi_category_pdf["aqi_category"]]


ax = aqi_category_pdf.plot(
    kind="bar",
    x="aqi_category",
    y="count",
    legend=False,
    figsize=(10, 4),
    color=colors,
)
ax.set_title("AQI Category Distribution")
ax.set_xlabel("AQI Category")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Pollutant Profile

In [ ]:
pollutants_pdf = report_tables["average_pollutants"]
display(pollutants_pdf)

In [ ]:
pollutant_values = pollutants_pdf.set_index("pollutant")["avg_value"]
ax = pollutant_values.plot(
    kind="bar",
    figsize=(8, 4),
    color=plt.cm.plasma(np.linspace(0, 1, len(pollutant_values))),
)
ax.set_title("Average Pollutant Levels")
ax.set_xlabel("Pollutant")
ax.set_ylabel("Average Value")
plt.tight_layout()
plt.show()

## Dominant Pollutants

In [ ]:
dominant_pdf = report_tables["dominant_pollutants"]
display(dominant_pdf)

In [ ]:
ax = dominant_pdf.plot(
    kind="bar",
    x="dominant_pollutant",
    y="count",
    legend=False,
    figsize=(8, 4),
    color=plt.cm.cividis(np.linspace(0, 1, len(dominant_pdf))),
)
ax.set_title("Dominant Pollutant Frequency")
ax.set_xlabel("Pollutant")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Weather Relationships

In [ ]:
correlations_pdf = report_tables["weather_correlations"]
display(correlations_pdf)

In [ ]:
correlation_values = correlations_pdf.iloc[0]
ax = correlation_values.plot(
    kind="bar", figsize=(8, 4), color=["tab:blue", "tab:orange", "tab:green"]
)
ax.set_title("AQI Correlations with Weather Metrics")
ax.set_xlabel("Metric")
ax.set_ylabel("Correlation")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
peak_hour = hourly_pdf.loc[hourly_pdf["avg_aqi"].idxmax()]
peak_day = daily_pdf.loc[daily_pdf["avg_aqi"].idxmax()]
top_category = aqi_category_pdf.iloc[0]
top_pollutant = dominant_pdf.iloc[0]
correlation_values = correlations_pdf.iloc[0].dropna()
strongest_metric = correlation_values.abs().idxmax()

print(
    f"Highest average hourly AQI: {peak_hour['avg_aqi']:.1f} at hour {int(peak_hour['hour'])}."
)
print(f"Highest average daily AQI: {peak_day['avg_aqi']:.1f} on {peak_day['day']}.")
print(
    f"Most common AQI category: {top_category['aqi_category']} ({int(top_category['count'])} records)."
)
print(
    f"Most common dominant pollutant: {top_pollutant['dominant_pollutant']} ({int(top_pollutant['count'])} records)."
)
print(
    f"Strongest AQI-weather relationship by absolute correlation: {strongest_metric} = {correlation_values[strongest_metric]:.3f}."
)